In [2]:
import sys

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from rag_hyde import HypotheticalDocumentGenerator

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
filename = Path("/home/tacktile-systems/SIM_Support-ChatBot/knowledge_base/RAG_KB_bot.md")

with open(filename, "r") as f:
    markdown_text = f.read()
    print("File Loaded Successfully")

File Loaded Successfully


In [4]:
# Configure header splitting rules
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_text)

In [6]:
len(md_header_splits)

407

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=60
)

chunks = text_splitter.split_documents(md_header_splits)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 568


---
# Embeddings and Vector Stores
---

In [8]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    collection_name="rag_hyde_collection",
)

print(f"Stored: {vectorstore._collection.count()} chunks in the vectorstore")

Stored: 568 chunks in the vectorstore


In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
    )

### HyDE Retriever Instantiation
- document_llm — used internally by CustomHypotheticalDocumentEmbedder to generate the hypothetical document from the query.
- generation_llm — used later in the RAG generation step to produce the final answer.

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

hyde_retriever = HypotheticalDocumentGenerator.from_llm(
    llm=llm,
    retriever=retriever
)

print("HyDE retriever is ready")

HyDE retriever is ready


### HyDE Retrieval
The query is passed to hyde_retriever.invoke(), which internally:

1. Generates a hypothetical document via document_llm
2. Embeds it and queries ChromaDB
3. Returns the top-3 most similar real documents

In [11]:
query = "How to delete invoice?"

retrieved_docs = hyde_retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"[{i}] | Source: {doc.metadata.get("Header 1")} | Topic: {doc.metadata.get("Header 2")}")
    print(doc.page_content)
    print()

[1] | Source: Invoices | Topic: Actions After Invoice Creation
*   **Make Delivery Note**: Generate a delivery note directly from the invoice.
*   **Create Duplicate**: Create an exact copy of the existing invoice without client.
*   **Make Sales Return**: Initiate a sales return related to the invoice.
*   **Delete**: Permanently remove the invoice.
*   **Cancel Invoice**: Invalidate the invoice.

[2] | Source: Invoices | Topic: Actions After Invoice Creation
After creating an invoice, you can perform various actions by clicking on the invoice from invoice list. These actions include:  
*   **Edit**: Modify invoice details such as images, notes, items, or names.
*   **Preview**: View the invoice in either image or PDF format.
*   **Send**: Dispatch the invoice using a template to any recipient.
*   **Share**: Directly share the invoice as an image or PDF.
*   **Print**: Print the invoice using either a Normal or Thermal Printer.
*   **Mark as Received**: Record the invoice as paid, sp

### Content Fusion

In [12]:
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

print("=== Fused Context ===")
print(context)

=== Fused Context ===
*   **Make Delivery Note**: Generate a delivery note directly from the invoice.
*   **Create Duplicate**: Create an exact copy of the existing invoice without client.
*   **Make Sales Return**: Initiate a sales return related to the invoice.
*   **Delete**: Permanently remove the invoice.
*   **Cancel Invoice**: Invalidate the invoice.

After creating an invoice, you can perform various actions by clicking on the invoice from invoice list. These actions include:  
*   **Edit**: Modify invoice details such as images, notes, items, or names.
*   **Preview**: View the invoice in either image or PDF format.
*   **Send**: Dispatch the invoice using a template to any recipient.
*   **Share**: Directly share the invoice as an image or PDF.
*   **Print**: Print the invoice using either a Normal or Thermal Printer.
*   **Mark as Received**: Record the invoice as paid, specifying the payment method (Cash or Bank).

**Q: Why should I not delete the original purchase invoice 

In [13]:
generation_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [17]:
generation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an expert internal support assistant for our organization. Your goal is to provide accurate, professional, and clear answers to user inquiries based strictly on the provided context.

### Instructions:
1. Grounding: Answer the user's question using ONLY the provided context. Do not use outside knowledge or assume facts not explicitly stated.
2. Gap Handling: If the context does not contain enough information to answer the question completely, state: "I cannot find that information in our current documentation."
3. Tone: Maintain a helpful, polite, professional, and constructive tone.
4. Formatting: 
   - Use a simple, sequential numbered list (1, 2, 3...) for multi-step instructions.
   - Strictly do NOT use nested bullets or multi-level indentations.
   - Use bolding only for key terms, interface items, and critical actions.
5. Conciseness: Keep each step limited to a single concise sentence. Eliminate conversational introductory remarks.
6. Conditional Video Links: If and only if the provided context contains a YouTube link (e.g., watch?v=...), append a single standalone line at the very end of your response formatted exactly like this: "🎥 **Video Tutorial:** [Watch here](URL)"

### Context:
{context}

### Question:
{query}

### Answer:

"""
    ),
    ("human", "Context:\n{context}\n\nQuestion: {query}")
])



generation_chain = generation_prompt | generation_llm
response = generation_chain.invoke({"context": context, "query": query})

print("=== Final Answer ===")
print(response.content)

=== Final Answer ===
I cannot find that information in our current documentation.


In [15]:
# Interactive loop (new cell) — builds context and passes correct variables
while True:
    query = input("\nYou: ")

    if query.lower() in ("exit", "bye", "ok", "thanks"):
        break

    docs = retriever.invoke(query)
    if not docs:
        print("No relevant documents found.")
        continue

    context = "\n\n".join([doc.page_content for doc in docs])

    answer = generation_chain.invoke({"context": context, "query": query})

    print(f"Query: {query}")
    print("Chatbot:\n",answer.content)
    print()
    print("="*30)


Query: I want to buy plan
Chatbot:
 1. Go to **Side Menu**.  
2. Select **Upgrade**.  
3. Click on **Purchase Subscription**.  
4. Choose a plan: **Monthly**, **Annual**, or **1-Year No Auto-Renew**.  
5. Proceed with **payment**.

Query: My free trail has been ended, now what to do?
Chatbot:
 1. You cannot create new invoices until you purchase a subscription.  
2. Go to **Side Menu**.  
3. Select **Upgrade**.  
4. Choose **Purchase Subscription**.  
5. Select a plan (Monthly, Annual, or 1-Year No Auto-Renew).  
6. Proceed with payment to restore full access.

Query: App is not working
Chatbot:
 1. Ensure all devices are using the **latest version** of the app.  
2. Check if you are attempting to upload **very large files** (1000+ records), which may slow down the app.  
3. If the app crashed or you experienced unexpected data loss, consider reinstalling the app.  
4. If you want to clear/reset your data without deleting your account, please **contact support**.  
5. If issues persist